# Task 1 - Financial AI: LLM-Powered Equity Research Assistant

Run each cell top to bottom in Google Colab. Requires a free Groq API key
(console.groq.com) stored as a Colab secret or environment variable
`GROQ_API_KEY` -- **never hardcode it in this notebook.**

In [ ]:
!pip install -q yfinance groq pydantic matplotlib pandas numpy

In [ ]:
import os
from google.colab import userdata
# Store your key via Colab's Secrets panel (key icon in the left sidebar),
# name it GROQ_API_KEY. This keeps it out of the notebook file entirely.
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

In [ ]:
# If running from the cloned repo, src/ is already alongside this notebook.
import sys
sys.path.append('src')

from data_pipeline import run_pipeline
from llm_reasoning import _get_client, classify_all_headlines, generate_trade_signal
from report_renderer import render_report

TICKER = 'AAPL'  # change to any ticker you like

## Task 1A - Data Pipeline

In [ ]:
result = run_pipeline(TICKER)
summary = result['summary']
ohlcv = result['ohlcv']
news = result['news']

print(summary)
print(f"\nRetrieved {len(news)} headlines")
ohlcv.tail()

## Task 1B - LLM Sentiment and Signal Reasoning

In [ ]:
client = _get_client()
headline_sentiments, agg_sentiment = classify_all_headlines(client, news)

for h in headline_sentiments:
    print(f"[{h.sentiment:>8}] conf={h.confidence:.2f}  {h.headline}")
print('\nAggregate:', agg_sentiment)

In [ ]:
latest = ohlcv.iloc[-1].to_dict()
latest['fifty_two_week_high'] = summary.fifty_two_week_high
latest['fifty_two_week_low'] = summary.fifty_two_week_low
latest['ytd_return_pct'] = summary.ytd_return_pct

trade_signal = generate_trade_signal(client, TICKER, latest, agg_sentiment)
print(trade_signal)

## Bonus - Rendered HTML Research Brief

In [ ]:
html = render_report(TICKER, summary, headline_sentiments, agg_sentiment, trade_signal, ohlcv)
with open(f'{TICKER}_research_brief.html', 'w') as f:
    f.write(html)

from IPython.display import HTML, display
display(HTML(html))